# Pass Tagging Tool

Tags departure and arrival frames for each pass, with manual ball relocation for gap frames.

**Workflow:**
1. Configure `VIDEO_PATH` and `TRACKS_CSV` in the next cell.
2. Run all cells — the interactive UI appears at the bottom.
3. Navigate to the departure frame (ball leaves foot) → click **Tag Departure**.
4. Navigate to the arrival frame (receiver makes contact) → click **Tag Arrival**.
5. If the ball marker is orange (interpolated) or missing, click **Relocate ball** and enter the pixel coordinates before tagging.
6. Click **Save events** when done. The CSV is written to `EVENTS_CSV`.
7. Run the final merge cell to produce `per_frame_tracks_with_passes.csv`.

**Pass timing convention:** departure = frame ball visibly leaves the kicker's foot; arrival = frame receiver makes first contact. Both frames are tagged; timestamps are derived automatically from `frame / fps`.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
VIDEO_PATH   = "/content/playbook/HILAL-HAZM_match_B_up8.mp4"  # source video
TRACKS_CSV   = "/content/playbook/output/per_frame_tracks.csv" # main.py output
EVENTS_CSV   = "/content/playbook/output/pass_events.csv"       # written here
MERGED_CSV   = "/content/playbook/output/per_frame_tracks_with_passes.csv"

# Class IDs (must match your model config)
BALL_CLASS   = 0
GK_CLASS     = 1
PLAYER_CLASS = 2
REF_CLASS    = 3

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import cv2, json, os
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display as ipy_display
from io import BytesIO
from PIL import Image as PILImage, ImageDraw, ImageFont

print("Imports OK")

In [ ]:
# ── Load tracking data ─────────────────────────────────────────────────────
df_tracks = pd.read_csv(TRACKS_CSV)

# Ball rows (one per frame at most, may have gaps)
df_ball = df_tracks[df_tracks["class_id"] == BALL_CLASS].copy()
df_ball["cx"] = (df_ball["x1"] + df_ball["x2"]) / 2
df_ball["cy"] = (df_ball["y1"] + df_ball["y2"]) / 2
ball_idx = df_ball.set_index("frame")

# Player rows (all non-ball classes)
df_players = df_tracks[df_tracks["class_id"] != BALL_CLASS].copy()
df_players["cx"] = (df_players["x1"] + df_players["x2"]) / 2
df_players["cy"] = (df_players["y1"] + df_players["y2"]) / 2
players_grp = df_players.groupby("frame")

# Video metadata
cap = cv2.VideoCapture(VIDEO_PATH)
FPS          = cap.get(cv2.CAP_PROP_FPS) or 25.0
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VID_W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VID_H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Pitch bounds (cm) from the track CSV if available
pitch_cols = ["x_m", "y_m"]
has_pitch  = all(c in df_tracks.columns for c in pitch_cols)

print(f"Video  : {TOTAL_FRAMES} frames @ {FPS:.2f} fps  ({VID_W}×{VID_H})")
print(f"Tracks : {len(df_tracks)} rows | ball frames: {len(df_ball)} | player frames: {len(df_players)}")
print(f"Pitch coords available: {has_pitch}")

In [ ]:
# ── Load or initialise events ──────────────────────────────────────────────
if os.path.exists(EVENTS_CSV):
    df_ev   = pd.read_csv(EVENTS_CSV)
    events  = df_ev.to_dict("records")
    next_id = int(df_ev["pass_id"].max()) + 1 if len(df_ev) else 1
    print(f"Loaded {len(events)} existing events from {EVENTS_CSV}")
else:
    events  = []
    next_id = 1
    print("No existing events file — starting fresh.")

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────

# Manual ball position overrides:  frame_idx -> (cx, cy)
ball_overrides: dict[int, tuple[float, float]] = {}

def read_frame(idx: int) -> np.ndarray | None:
    """Return RGB frame at idx, or None."""
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, bgr = cap.read()
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if ok else None


def get_ball(idx: int) -> tuple[float | None, float | None, str]:
    """
    Returns (cx, cy, source) where source is one of:
      'manual'    – user override
      'tracker'   – real detection
      'interp'    – smoother-extrapolated
      'missing'   – not in CSV at all
    """
    if idx in ball_overrides:
        cx, cy = ball_overrides[idx]
        return cx, cy, "manual"
    if idx in ball_idx.index:
        row = ball_idx.loc[idx]
        if isinstance(row, pd.DataFrame):   # duplicate frame rows → take first
            row = row.iloc[0]
        interp = bool(row.get("ball_interpolated", False))
        return float(row.cx), float(row.cy), "interp" if interp else "tracker"
    return None, None, "missing"


def get_pitch_xy(idx: int) -> tuple[float | None, float | None]:
    """Pitch coordinates of the ball at idx (cm), or (None, None)."""
    if not has_pitch or idx not in ball_idx.index:
        return None, None
    row = ball_idx.loc[idx]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    xm = row.get("x_m", None)
    ym = row.get("y_m", None)
    try:
        return float(xm), float(ym)
    except (TypeError, ValueError):
        return None, None


def nearest_player(idx: int, cx: float, cy: float) -> tuple[int | None, float]:
    """(display_track_id, distance_px) of the player closest to (cx, cy)."""
    if idx not in players_grp.groups:
        return None, float("inf")
    grp = players_grp.get_group(idx)
    if grp.empty:
        return None, float("inf")
    dists = np.hypot(grp.cx - cx, grp.cy - cy)
    best  = dists.idxmin()
    return int(grp.loc[best, "display_track_id"]), float(dists[best])


# Team colour palette (team_id 0=pink, 1=cyan, unknown=white, ref=orange)
_TEAM_COLOURS = {0: (255, 105, 180), 1: (0, 200, 255), -1: (200, 200, 200)}

def _player_colour(row):
    if row.get("class_id") == REF_CLASS:
        return (255, 140, 0)
    tid = int(row.get("team_id", -1)) if pd.notna(row.get("team_id", None)) else -1
    return _TEAM_COLOURS.get(tid, (200, 200, 200))


def annotate(frame: np.ndarray, idx: int,
             dep_info: dict | None = None,
             arr_info: dict | None = None) -> bytes:
    """
    Draw players, ball and pending pass markers on frame.
    Returns JPEG bytes for the ipywidgets Image widget.
    """
    img = PILImage.fromarray(frame)
    d   = ImageDraw.Draw(img)
    W, H = img.size
    scale = W / 1280  # normalise stroke width to a 1280-wide reference

    # ── Players ────────────────────────────────────────────────────────────
    if idx in players_grp.groups:
        for _, row in players_grp.get_group(idx).iterrows():
            x1, y1, x2, y2 = int(row.x1), int(row.y1), int(row.x2), int(row.y2)
            col = _player_colour(row)
            lw  = max(1, int(2 * scale))
            d.rectangle([x1, y1, x2, y2], outline=col, width=lw)
            label = f"P{int(row.display_track_id)}"
            d.text((x1, max(0, y1 - 14)), label, fill=col)

    # ── Ball ───────────────────────────────────────────────────────────────
    cx, cy, src = get_ball(idx)
    if cx is not None:
        r   = max(8, int(10 * scale))
        lw  = max(2, int(3 * scale))
        col = {"tracker": (255, 230, 0),
               "interp":  (255, 140, 0),
               "manual":  (0, 255, 120)}.get(src, (200, 200, 200))
        d.ellipse([cx - r, cy - r, cx + r, cy + r], outline=col, width=lw)
        tag = {"tracker": "BALL", "interp": "BALL(interp)",
               "manual": "BALL(manual)", "missing": ""}.get(src, "")
        if tag:
            d.text((cx + r + 3, cy - 8), tag, fill=col)

    # ── Departure marker ───────────────────────────────────────────────────
    if dep_info:
        dx, dy = dep_info["dep_ball_px"], dep_info["dep_ball_py"]
        r2 = max(12, int(16 * scale))
        d.ellipse([dx - r2, dy - r2, dx + r2, dy + r2], outline=(0, 255, 80), width=3)
        d.text((dx + r2 + 3, dy - 10), f"DEP P{dep_info['passer_id']}", fill=(0, 255, 80))

    # ── Arrival marker ─────────────────────────────────────────────────────
    if arr_info:
        ax, ay = arr_info["arr_ball_px"], arr_info["arr_ball_py"]
        r2 = max(12, int(16 * scale))
        d.ellipse([ax - r2, ay - r2, ax + r2, ay + r2], outline=(80, 180, 255), width=3)
        d.text((ax + r2 + 3, ay - 10), f"ARR P{arr_info['receiver_id']}", fill=(80, 180, 255))

    # ── Frame info ─────────────────────────────────────────────────────────
    ts = idx / FPS
    d.text((8, 8), f"Frame {idx}  |  {ts:.2f}s", fill=(255, 255, 255))

    buf = BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return buf.getvalue()


print("Helpers ready.")

In [ ]:
# ── Interactive tagging UI ─────────────────────────────────────────────────

# ── Mutable state living outside widgets (avoids circular observe issues) ──
state = {
    "frame":   0,
    "pending": {},       # partial pass being built
    "events":  events,   # finalised pass list
    "next_id": next_id,
}

# ── Widgets ────────────────────────────────────────────────────────────────
frame_img   = widgets.Image(format="jpeg", layout=widgets.Layout(width="100%"))

w_slider    = widgets.IntSlider(
    min=0, max=TOTAL_FRAMES - 1, step=1, value=0,
    description="Frame:", continuous_update=False,
    layout=widgets.Layout(width="90%"),
)
w_step      = widgets.BoundedIntText(value=1, min=1, max=100,
                                      description="Step:",
                                      layout=widgets.Layout(width="100px"))
w_prev      = widgets.Button(description="◀ Prev",  button_style="",
                              layout=widgets.Layout(width="80px"))
w_next      = widgets.Button(description="Next ▶",  button_style="",
                              layout=widgets.Layout(width="80px"))
w_goto_lbl  = widgets.Label("Go to frame:")
w_goto_val  = widgets.BoundedIntText(value=0, min=0, max=TOTAL_FRAMES - 1,
                                      layout=widgets.Layout(width="90px"))
w_goto_btn  = widgets.Button(description="Go", layout=widgets.Layout(width="50px"))

w_ball_info = widgets.HTML(value="<i>Ball info</i>")
w_status    = widgets.HTML(value="<b>Status:</b> idle")

# ── Ball relocation panel ──────────────────────────────────────────────────
w_rel_cx    = widgets.FloatText(description="Ball CX:", layout=widgets.Layout(width="160px"))
w_rel_cy    = widgets.FloatText(description="Ball CY:", layout=widgets.Layout(width="160px"))
w_rel_set   = widgets.Button(description="Set ball pos", button_style="warning",
                              layout=widgets.Layout(width="120px"))
w_rel_clear = widgets.Button(description="Clear override", button_style="",
                              layout=widgets.Layout(width="130px"))
w_rel_info  = widgets.HTML("")
reloc_box   = widgets.HBox([w_rel_cx, w_rel_cy, w_rel_set, w_rel_clear, w_rel_info])

# ── Tagging buttons ────────────────────────────────────────────────────────
w_tag_dep   = widgets.Button(description="Tag Departure", button_style="success",
                              layout=widgets.Layout(width="140px"))
w_tag_arr   = widgets.Button(description="Tag Arrival",   button_style="info",
                              layout=widgets.Layout(width="130px"))
w_cancel    = widgets.Button(description="Cancel pass",   button_style="warning",
                              layout=widgets.Layout(width="120px"))
w_del_last  = widgets.Button(description="Delete last",   button_style="danger",
                              layout=widgets.Layout(width="110px"))
w_save      = widgets.Button(description="💾 Save events", button_style="primary",
                              layout=widgets.Layout(width="130px"))

# ── Outcome selector (shown after arrival is tagged) ──────────────────────
w_outcome   = widgets.Dropdown(
    options=[("Complete", "complete"), ("Incomplete", "incomplete"),
             ("Unknown", "unknown")],
    value="complete", description="Outcome:",
    layout=widgets.Layout(width="180px"),
)
w_notes     = widgets.Text(description="Notes:", placeholder="optional",
                            layout=widgets.Layout(width="260px"))
w_confirm   = widgets.Button(description="✔ Confirm pass", button_style="success",
                              layout=widgets.Layout(width="140px"))
confirm_row = widgets.HBox([w_outcome, w_notes, w_confirm])
confirm_row.layout.display = "none"

# ── Events table display ───────────────────────────────────────────────────
w_table     = widgets.Output()

# ── Refresh helper ─────────────────────────────────────────────────────────
def refresh(idx=None):
    if idx is None:
        idx = state["frame"]
    rgb = read_frame(idx)
    if rgb is None:
        return

    p   = state["pending"]
    dep = p if "dep_frame" in p else None
    arr = p if "arr_frame" in p else None

    frame_img.value = annotate(rgb, idx, dep_info=dep, arr_info=arr)

    # Ball info line
    cx, cy, src = get_ball(idx)
    src_col = {"tracker": "#cce", "interp": "orange",
               "manual": "#8f8", "missing": "red"}.get(src, "white")
    if cx is not None:
        xm, ym = get_pitch_xy(idx)
        pitch_str = f"  pitch ({xm:.0f}, {ym:.0f}) cm" if xm is not None else ""
        ball_html = (f"Ball: <b style='color:{src_col}'>{src}</b>  "
                     f"px=({cx:.0f}, {cy:.0f}){pitch_str}")
    else:
        ball_html = f"<b style='color:red'>Ball: missing — relocate before tagging</b>"
    w_ball_info.value = ball_html

    # Prefill relocation inputs with current ball position
    if cx is not None:
        w_rel_cx.value = round(cx, 1)
        w_rel_cy.value = round(cy, 1)

    # Status
    if not p:
        w_status.value = "<b>Status:</b> idle — navigate to the kick frame, then press <b>Tag Departure</b>"
    elif "dep_frame" in p and "arr_frame" not in p:
        w_status.value = (f"<b>Status:</b> departure tagged at frame "
                          f"<b>{p['dep_frame']}</b> (passer P{p['passer_id']}) "
                          f"— navigate to arrival frame, then press <b>Tag Arrival</b>")
    elif "arr_frame" in p:
        w_status.value = (f"<b>Status:</b> both frames tagged. "
                          f"Set outcome &amp; notes, then press <b>✔ Confirm pass</b>")


def refresh_table():
    with w_table:
        w_table.clear_output(wait=True)
        if state["events"]:
            ipy_display(pd.DataFrame(state["events"])[
                ["pass_id", "departure_frame", "departure_time",
                 "passer_id", "arrival_frame", "arrival_time",
                 "receiver_id", "outcome"]
            ].tail(10))
        else:
            print("No events yet.")


# ── Navigation callbacks ───────────────────────────────────────────────────
def _set_frame(idx):
    idx = max(0, min(TOTAL_FRAMES - 1, idx))
    state["frame"] = idx
    w_slider.value = idx
    refresh(idx)

def on_slider(change):
    state["frame"] = change["new"]
    refresh(change["new"])

def on_prev(_):
    _set_frame(state["frame"] - w_step.value)

def on_next(_):
    _set_frame(state["frame"] + w_step.value)

def on_goto(_):
    _set_frame(w_goto_val.value)

w_slider.observe(on_slider, names="value")
w_prev.on_click(on_prev)
w_next.on_click(on_next)
w_goto_btn.on_click(on_goto)


# ── Relocation callbacks ───────────────────────────────────────────────────
def on_set_ball(_):
    idx = state["frame"]
    ball_overrides[idx] = (w_rel_cx.value, w_rel_cy.value)
    w_rel_info.value = f"<span style='color:#8f8'>Override set for frame {idx}</span>"
    refresh(idx)

def on_clear_ball(_):
    idx = state["frame"]
    ball_overrides.pop(idx, None)
    w_rel_info.value = f"<span style='color:orange'>Override cleared for frame {idx}</span>"
    refresh(idx)

w_rel_set.on_click(on_set_ball)
w_rel_clear.on_click(on_clear_ball)


# ── Tagging callbacks ──────────────────────────────────────────────────────
def on_tag_dep(_):
    idx   = state["frame"]
    cx, cy, src = get_ball(idx)
    if cx is None:
        w_status.value = ("<b style='color:red'>No ball at this frame.</b> "
                          "Use the Relocate panel to set a manual position first.")
        return
    pid, dist = nearest_player(idx, cx, cy)
    xm, ym = get_pitch_xy(idx)
    state["pending"] = {
        "dep_frame":    idx,
        "dep_time":     round(idx / FPS, 3),
        "dep_ball_px":  cx,
        "dep_ball_py":  cy,
        "dep_ball_xm":  xm,
        "dep_ball_ym":  ym,
        "dep_src":      src,
        "passer_id":    pid,
        "passer_dist_px": round(dist, 1),
    }
    confirm_row.layout.display = "none"
    refresh(idx)


def on_tag_arr(_):
    if "dep_frame" not in state["pending"]:
        w_status.value = "<b style='color:red'>Tag Departure first.</b>"
        return
    idx   = state["frame"]
    cx, cy, src = get_ball(idx)
    if cx is None:
        w_status.value = ("<b style='color:red'>No ball at this frame.</b> "
                          "Use the Relocate panel to set a manual position first.")
        return
    pid, dist = nearest_player(idx, cx, cy)
    xm, ym = get_pitch_xy(idx)
    state["pending"].update({
        "arr_frame":    idx,
        "arr_time":     round(idx / FPS, 3),
        "arr_ball_px":  cx,
        "arr_ball_py":  cy,
        "arr_ball_xm":  xm,
        "arr_ball_ym":  ym,
        "arr_src":      src,
        "receiver_id":  pid,
        "receiver_dist_px": round(dist, 1),
    })
    confirm_row.layout.display = ""
    refresh(idx)


def on_confirm(_):
    p = state["pending"]
    if "dep_frame" not in p or "arr_frame" not in p:
        return
    record = {
        "pass_id":          state["next_id"],
        "departure_frame":  p["dep_frame"],
        "departure_time":   p["dep_time"],
        "passer_id":        p["passer_id"],
        "passer_dist_px":   p["passer_dist_px"],
        "ball_dep_px":      round(p["dep_ball_px"], 1),
        "ball_dep_py":      round(p["dep_ball_py"], 1),
        "ball_dep_xm":      p["dep_ball_xm"],
        "ball_dep_ym":      p["dep_ball_ym"],
        "dep_src":          p["dep_src"],
        "arrival_frame":    p["arr_frame"],
        "arrival_time":     p["arr_time"],
        "receiver_id":      p["receiver_id"],
        "receiver_dist_px": p["receiver_dist_px"],
        "ball_arr_px":      round(p["arr_ball_px"], 1),
        "ball_arr_py":      round(p["arr_ball_py"], 1),
        "ball_arr_xm":      p["arr_ball_xm"],
        "ball_arr_ym":      p["arr_ball_ym"],
        "arr_src":          p["arr_src"],
        "duration_frames":  p["arr_frame"] - p["dep_frame"],
        "duration_s":       round((p["arr_frame"] - p["dep_frame"]) / FPS, 3),
        "outcome":          w_outcome.value,
        "notes":            w_notes.value.strip(),
    }
    state["events"].append(record)
    state["next_id"] += 1
    state["pending"] = {}
    confirm_row.layout.display = "none"
    w_notes.value = ""
    refresh_table()
    refresh()


def on_cancel(_):
    state["pending"] = {}
    confirm_row.layout.display = "none"
    refresh()


def on_del_last(_):
    if state["events"]:
        removed = state["events"].pop()
        state["next_id"] = removed["pass_id"]
        refresh_table()
        refresh()


def on_save(_):
    df_save = pd.DataFrame(state["events"])
    os.makedirs(os.path.dirname(EVENTS_CSV), exist_ok=True)
    df_save.to_csv(EVENTS_CSV, index=False)
    w_status.value = (f"<b style='color:#8f8'>✔ Saved {len(state['events'])} events "
                      f"→ {EVENTS_CSV}</b>")


w_tag_dep.on_click(on_tag_dep)
w_tag_arr.on_click(on_tag_arr)
w_confirm.on_click(on_confirm)
w_cancel.on_click(on_cancel)
w_del_last.on_click(on_del_last)
w_save.on_click(on_save)


# ── Layout ─────────────────────────────────────────────────────────────────
nav_row  = widgets.HBox([w_prev, w_next, w_step,
                          w_goto_lbl, w_goto_val, w_goto_btn])
tag_row  = widgets.HBox([w_tag_dep, w_tag_arr, w_cancel,
                          w_del_last, w_save])
reloc_section = widgets.VBox([
    widgets.HTML("<b>Relocate ball (use when ball is missing or interpolated):</b>"),
    reloc_box,
])
ui = widgets.VBox([
    frame_img,
    w_slider,
    nav_row,
    w_ball_info,
    reloc_section,
    tag_row,
    confirm_row,
    w_status,
    widgets.HTML("<hr><b>Last 10 tagged passes:</b>"),
    w_table,
])

ipy_display(ui)
refresh_table()
refresh(0)

In [ ]:
# ── Merge events with per-frame tracks ─────────────────────────────────────
#
# Adds three columns to every track row:
#   pass_id       – which pass this row belongs to (NaN if none)
#   pass_role     – 'passer' | 'receiver' | NaN
#   pass_phase    – 'departure' | 'arrival' | 'in_flight' | NaN
#
# The ball rows between departure and arrival frames are tagged 'in_flight'.
# Run this cell after tagging is complete (or any time to preview).

def merge_events(df_tracks: pd.DataFrame,
                 events: list[dict]) -> pd.DataFrame:
    df = df_tracks.copy()
    df["pass_id"]    = pd.NA
    df["pass_role"]  = pd.NA
    df["pass_phase"] = pd.NA

    for ev in events:
        pid        = ev["pass_id"]
        dep_f      = ev["departure_frame"]
        arr_f      = ev["arrival_frame"]
        passer_tid = ev.get("passer_id")
        recv_tid   = ev.get("receiver_id")

        # Ball rows between departure and arrival → in_flight
        ball_flight = (
            (df["class_id"] == BALL_CLASS)
            & (df["frame"] >= dep_f)
            & (df["frame"] <= arr_f)
        )
        df.loc[ball_flight, "pass_id"]    = pid
        df.loc[ball_flight, "pass_phase"] = "in_flight"

        # Passer at departure frame
        if passer_tid is not None:
            passer_row = (
                (df["frame"] == dep_f)
                & (df["display_track_id"] == passer_tid)
            )
            df.loc[passer_row, "pass_id"]    = pid
            df.loc[passer_row, "pass_role"]  = "passer"
            df.loc[passer_row, "pass_phase"] = "departure"

        # Receiver at arrival frame
        if recv_tid is not None:
            recv_row = (
                (df["frame"] == arr_f)
                & (df["display_track_id"] == recv_tid)
            )
            df.loc[recv_row, "pass_id"]    = pid
            df.loc[recv_row, "pass_role"]  = "receiver"
            df.loc[recv_row, "pass_phase"] = "arrival"

    return df


df_merged = merge_events(df_tracks, state["events"])
df_merged.to_csv(MERGED_CSV, index=False)

tagged = df_merged[df_merged["pass_id"].notna()]
print(f"Merged CSV written → {MERGED_CSV}")
print(f"Tagged rows: {len(tagged)} / {len(df_merged)} total")
print(f"Passes: {len(state['events'])}")
if state["events"]:
    df_ev = pd.DataFrame(state["events"])
    print("\nPass summary:")
    ipy_display(df_ev[["pass_id","departure_frame","departure_time",
                        "passer_id","arrival_frame","arrival_time",
                        "receiver_id","duration_s","outcome"]])

In [ ]:
# ── Quick stats (optional) ─────────────────────────────────────────────────
if state["events"]:
    df_ev = pd.DataFrame(state["events"])
    print(f"Total passes tagged : {len(df_ev)}")
    print(f"Complete            : {(df_ev.outcome == 'complete').sum()}")
    print(f"Incomplete          : {(df_ev.outcome == 'incomplete').sum()}")
    print(f"Completion rate     : {(df_ev.outcome == 'complete').mean():.1%}")
    print(f"Avg duration        : {df_ev.duration_s.mean():.2f}s")
    print(f"\nPasses per passer:")
    ipy_display(df_ev.groupby("passer_id").size().rename("passes").sort_values(ascending=False))
    print(f"\nBall source at departure:")
    ipy_display(df_ev.groupby("dep_src").size())
else:
    print("No events tagged yet.")